# BatSpot Fine-Tuning on Kaggle — Detector + Classifier Pipeline

Fine-tune BatSpot pre-trained models on custom bat vocalisation data.

**Pipeline:**
1. **Setup** — Config, dependencies, model/dataset definitions
2. **Data Prep** — Load data, create detector (binary) and classifier (8-class) datasets, resample
3. **Pretrained Models** — Load detector.pk (192kHz) + classifier.pk (250kHz) from Kaggle upload
4. **Detector Training** — Fine-tune binary detector, save detector.pk, evaluate
5. **Classifier Training** — Fine-tune 8-class classifier, save classifier.pk, evaluate
6. **Combined Pipeline** — Detector → filter → classifier, combined metrics
7. **Export** — Save both .pk files
8. **Existing Model Validation** — Evaluate all models in Data/model_output/

In [ ]:
# Cell 2: Configuration
import os

# --- Paths (Kaggle defaults, override as needed) ---
DATA_DIR = '/kaggle/input/batspot-data/final_dataset/data'  # Pre-extracted clips (class subfolders)
RAW_AUDIO_DIR = None                                         # Raw .WAV recordings (for clip extraction)
SELECTIONS_DIR = None                                        # Raven selection tables (for clip extraction)

# Clip extraction re-reads ALL raw audio and writes into DATA_DIR. You already
# have the extracted clips in final_dataset/data, so leave this False unless
# you specifically want to rebuild them from the raw recordings.
RUN_CLIP_EXTRACTION = False
OVERWRITE_EXISTING_CLIPS = False   # guard: refuse to write into a non-empty DATA_DIR
MODEL_OUTPUT_DIR = None                                      # Your own previously-trained models (validation)

# --- Auto-detect MODEL_OUTPUT_DIR relative to DATA_DIR ---
if MODEL_OUTPUT_DIR is None and DATA_DIR is not None:
    _data_parent = os.path.dirname(os.path.abspath(DATA_DIR))
    for _candidate in [
        os.path.join(_data_parent, 'model_output'),
        os.path.join(_data_parent, '..', 'model_output'),
        os.path.join(_data_parent, '..', '..', 'model_output'),
        'Data/model_output',
    ]:
        _candidate = os.path.normpath(_candidate)
        if os.path.isdir(_candidate):
            MODEL_OUTPUT_DIR = _candidate
            break

# ---------------------------------------------------------------------------
# OFFICIAL PRE-TRAINED MODELS -- set paths manually here.
#
# Fill in the 3 call detectors + 1 call classifier you uploaded as a Kaggle
# dataset. Leave any of them as None to skip it (Cell 10 falls back to
# auto-discovery under /kaggle/input/*, then to Data/model_output/).
#
# Click "Add Data" in Kaggle, then use "Copy Path" on the dataset folder to
# get the exact /kaggle/input/... prefix.
#
#   Detector   192 kHz, 2 classes (noise/target)  -> fine-tunes the detector
#   Classifier 250 kHz, 15 classes                -> fine-tunes the classifier
# ---------------------------------------------------------------------------
DETECTOR_M03_PATH = None    # e.g. '/kaggle/input/mymodels/batspot/models_call_detector/m03/train/ANIMAL-SPOT.pk'
DETECTOR_M09_PATH = None    # e.g. '/kaggle/input/mymodels/batspot/models_call_detector/m09/train/ANIMAL-SPOT.pk'
DETECTOR_M11_PATH = None    # e.g. '/kaggle/input/mymodels/batspot/models_call_detector/m11/train/ANIMAL-SPOT.pk'
CLASSIFIER_M09_PATH = None  # e.g. '/kaggle/input/mymodels/batspot/models_call_classifier/m09/train/ANIMAL-SPOT.pk'

# Manual paths take priority over auto-discovery. Set to False to ignore them.
USE_MANUAL_MODEL_PATHS = True

WORKING_DIR = '/kaggle/working'
PRETRAINED_DIR = os.path.join(WORKING_DIR, 'pretrained')
os.makedirs(PRETRAINED_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# Spectrogram params below are the REAL values read out of the official
# BatSpot .pk files (BatSpot_article/batspot/models_*/**/ANIMAL-SPOT.pk).
# They MUST match the model you fine-tune from, otherwise the transferred
# encoder weights were trained on a different time-frequency resolution and
# are effectively noise. Cell 10 re-verifies this against each .pk at runtime.
# ---------------------------------------------------------------------------

# --- Detector config (192 kHz, binary: noise vs target) ---
DET_CONFIG = {
    'sr': 192000,
    'num_classes': 2,
    'classes': ['noise', 'target'],
    'fmin': 1000,               # from official detector dataOpts
    'fmax': 95000,
    'n_fft': 256,
    'hop_length': 128,
    'n_freq_bins': 256,
    'sequence_len': 20,         # ms
    'max_pool': 2,
    'freq_compression': 'linear',
    'batch_size': 128,          # T4 x2: larger batch keeps both GPUs fed
    'base_lr': 1e-4,            # absolute LR — NOT scaled by batch size
    'n_epochs': 100,
    'epochs_per_eval': 2,
    'grad_accum_steps': 1,      # no accumulation needed with batch_size=128
    'use_multi_gpu': True,
    # reference hyperparams from models_call_detector/*/config_training_gui
    'beta1': 0.5,
    'lr_decay_factor': 0.5,
    'lr_patience_epochs': 8,
    'early_stopping_patience_epochs': 20,
}

# --- Classifier config (250 kHz, multi-species) ---
CLS_CONFIG = {
    'sr': 250000,
    'num_classes': None,        # auto-detected from your data in Cell 7
    'fmin': 10000,              # from official classifier dataOpts
    'fmax': 125000,
    'n_fft': 256,
    'hop_length': 128,
    'n_freq_bins': 256,
    'sequence_len': 20,         # ms
    'max_pool': 2,
    'freq_compression': 'linear',
    'batch_size': 128,
    'base_lr': 1e-4,            # absolute LR — NOT scaled by batch size
    'n_epochs': 150,            # paper config_train: max_train_epochs=150
    'epochs_per_eval': 2,
    'grad_accum_steps': 1,      # no accumulation needed with batch_size=128
    'use_multi_gpu': True,
    # reference hyperparams from models_call_classifier/m09/config_train
    'beta1': 0.5,
    'lr_decay_factor': 0.5,
    'lr_patience_epochs': 8,
    'early_stopping_patience_epochs': 20,
}

# --- Augmentation toggle ---
# True  = time shift + gaussian noise + freq/time masking applied to train splits
# False = no augmentation (faster, useful for debugging or ablation)
USE_AUGMENTATION = True

# --- GPU check ---
# If DEVICE prints 'cpu', the Kaggle GPU accelerator is OFF.
# To enable: Session options (right sidebar) -> Accelerator -> GPU T4 x2 -> Save.
# The notebook must then be restarted. No code change is needed.
DEVICE = __import__('torch').device('cuda' if __import__('torch').cuda.is_available() else 'cpu')

print(f'Detector:   {DET_CONFIG["sr"]//1000}kHz, {DET_CONFIG["num_classes"]} classes '
      f'(noise/target), fmin={DET_CONFIG["fmin"]}, fmax={DET_CONFIG["fmax"]}')
print(f'Classifier: {CLS_CONFIG["sr"]//1000}kHz, {CLS_CONFIG["num_classes"]} classes (auto), '
      f'fmin={CLS_CONFIG["fmin"]}, fmax={CLS_CONFIG["fmax"]}')
print(f'Device: {DEVICE}')

# Report which manual paths resolved (a typo here is otherwise silent)
if USE_MANUAL_MODEL_PATHS:
    print('\nManual pre-trained model paths:')
    for _lbl, _p in [('detector m03', DETECTOR_M03_PATH),
                     ('detector m09', DETECTOR_M09_PATH),
                     ('detector m11', DETECTOR_M11_PATH),
                     ('classifier m09', CLASSIFIER_M09_PATH)]:
        if _p is None:
            print(f'  {_lbl:15s} (not set)')
        elif os.path.isfile(_p):
            print(f'  {_lbl:15s} OK  {_p}')
        else:
            print(f'  {_lbl:15s} !! NOT FOUND: {_p}')
else:
    print('\nManual model paths DISABLED (USE_MANUAL_MODEL_PATHS = False)')


In [ ]:
# Cell 3: Install dependencies
__omp_shell("pip install -q resampy soundfile tqdm")

import resource
try:
    resource.setrlimit(resource.RLIMIT_NOFILE, (65535, 65535))
except Exception:
    pass

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name} ({props.total_memory / 1e9:.1f} GB)')


In [ ]:
# Cell 4: Imports
import os, sys, glob, json, shutil, zipfile, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from collections import OrderedDict, defaultdict, Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, roc_auc_score
)
import matplotlib.pyplot as plt
import resampy
import soundfile as sf
from tqdm.auto import tqdm
from math import ceil

sys.path.insert(0, os.path.abspath('.'))

print('All imports OK')

In [ ]:
# Cell 5: Model Architecture & Loading
# --- Faithful port of animal_spot/models/ (GPL-3.0, Bergler & Schroeter) ---
# Module/attribute names MUST match the original, or saved .pk state_dicts
# will not load. In particular the residual projection is `shortcut`
# (ResidualBase assigns self.shortcut = downsample), NOT `downsample`.

def get_padding(kernel_size):
    """Return `same` padding for a given kernel size."""
    if isinstance(kernel_size, int):
        return kernel_size // 2
    return tuple(s // 2 for s in kernel_size)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_ch, out_ch, stride=1, downsample=None, upsample=None, mid_ch=None):
        super().__init__()
        if mid_ch is None:
            mid_ch = out_ch
        if downsample is not None and upsample is not None:
            raise ValueError("Either downsample or upsample has to be None")

        if upsample is None:
            self.shortcut = downsample
            self.conv1 = nn.Conv2d(in_ch, mid_ch, kernel_size=3, stride=stride,
                                   padding=get_padding(3), bias=False)
        else:
            self.shortcut = upsample
            self.conv1 = nn.ConvTranspose2d(in_ch, mid_ch, kernel_size=3, stride=stride,
                                            padding=get_padding(3),
                                            output_padding=get_padding(stride), bias=False)
        self.bn1 = nn.BatchNorm2d(mid_ch)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(mid_ch, out_ch, kernel_size=3, stride=1,
                               padding=get_padding(3), bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu2 = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.shortcut is not None:
            residual = self.shortcut(x)
        out += residual
        out = self.relu2(out)
        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_ch, out_ch, stride=1, downsample=None, upsample=None, mid_ch=None):
        super().__init__()
        if mid_ch is None:
            mid_ch = out_ch
        if downsample is not None and upsample is not None:
            raise ValueError("Either downsample or upsample has to be None")
        self.shortcut = None
        if upsample is not None or downsample is not None:
            self.shortcut = downsample if downsample is not None else upsample

        self.conv1 = nn.Conv2d(in_ch, mid_ch, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_ch)
        self.relu1 = nn.ReLU(inplace=True)
        if upsample is not None:
            self.conv2 = nn.ConvTranspose2d(mid_ch, mid_ch, kernel_size=3, stride=stride,
                                            padding=get_padding(3),
                                            output_padding=get_padding(stride), bias=False)
        else:
            self.conv2 = nn.Conv2d(mid_ch, mid_ch, kernel_size=3, stride=stride,
                                   padding=get_padding(3), bias=False)
        self.bn2 = nn.BatchNorm2d(mid_ch)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv3 = nn.Conv2d(mid_ch, out_ch * self.expansion, kernel_size=1,
                               stride=1, padding=get_padding(1), bias=False)
        self.bn3 = nn.BatchNorm2d(out_ch * self.expansion)
        self.relu3 = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        out = self.relu1(self.bn1(self.conv1(x)))
        out = self.relu2(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.shortcut is not None:
            residual = self.shortcut(x)
        out += residual
        out = self.relu3(out)
        return out


def get_block_sizes(resnet_size):
    if resnet_size == 18:
        return [2, 2, 2, 2]
    elif resnet_size == 34:
        return [3, 4, 6, 3]
    elif resnet_size in (50, 101):
        return [3, 4, 23, 3]
    elif resnet_size == 152:
        return [3, 8, 36, 3]
    raise ValueError("Unsuported resnet size: ", resnet_size)


def get_block_type(resnet_size):
    if resnet_size in (18, 34):
        return BasicBlock
    elif resnet_size in (50, 101, 152):
        return Bottleneck
    raise ValueError("Unsuported resnet size ", resnet_size)


class ResidualBase(nn.Module):
    def __init__(self):
        super().__init__()
        self.cur_in_ch = 64

    def make_layer(self, block, out_ch, size, stride=1, shortcut="downsample"):
        layers = []
        stride_mean = stride
        if isinstance(stride, tuple):
            stride_mean = sum(stride) / len(stride)

        if shortcut == "upsample" and (stride_mean > 1
                                       or self.cur_in_ch != out_ch * block.expansion):
            shortcut = nn.ConvTranspose2d(self.cur_in_ch, out_ch * block.expansion,
                                          kernel_size=3, stride=stride,
                                          padding=get_padding(3),
                                          output_padding=get_padding(stride), bias=False)
            layers.append(block(self.cur_in_ch, out_ch, stride=stride,
                                mid_ch=self.cur_in_ch // block.expansion,
                                upsample=shortcut))
        elif shortcut == "downsample" and (stride_mean > 1
                                           or self.cur_in_ch != out_ch * block.expansion):
            shortcut = nn.Sequential(
                nn.Conv2d(self.cur_in_ch, out_ch * block.expansion, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_ch * block.expansion),
            )
            layers.append(block(self.cur_in_ch, out_ch, stride, downsample=shortcut))
        else:
            layers.append(block(self.cur_in_ch, out_ch))
        self.cur_in_ch = out_ch * block.expansion

        for _ in range(1, size):
            layers.append(block(self.cur_in_ch, out_ch))
        return nn.Sequential(*layers)


class ResidualEncoder(ResidualBase):
    def __init__(self, opts=None):
        super().__init__()
        if opts is None:
            opts = {"input_channels": 1, "conv_kernel_size": 7,
                    "max_pool": 1, "resnet_size": 18}
        self._opts = opts
        self.cur_in_ch = 64
        self.block_sizes = get_block_sizes(opts["resnet_size"])
        self.block_type = get_block_type(opts["resnet_size"])

        self.conv1 = nn.Conv2d(opts["input_channels"], out_channels=64,
                               kernel_size=opts["conv_kernel_size"], stride=(2, 2),
                               padding=get_padding(opts["conv_kernel_size"]), bias=False)
        self.bn1 = nn.BatchNorm2d(self.cur_in_ch)
        self.relu1 = nn.ReLU(inplace=True)
        if opts["max_pool"] == 1:
            self.max_pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=get_padding(3))
            stride1 = (1, 1)
        elif opts["max_pool"] == 0:
            self.max_pool = None
            stride1 = (2, 2)
        elif opts["max_pool"] == 2:
            self.max_pool = None
            stride1 = (1, 1)
        else:
            raise ValueError("Unknown max_pool option")

        self.layer1 = self.make_layer(self.block_type, 64, self.block_sizes[0], stride1)
        self.layer2 = self.make_layer(self.block_type, 128, self.block_sizes[1], (2, 2))
        self.layer3 = self.make_layer(self.block_type, 256, self.block_sizes[2], (2, 2))
        self.layer4 = self.make_layer(self.block_type, 512, self.block_sizes[3], (2, 2))

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        if self.max_pool is not None:
            x = self.max_pool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x

    def model_opts(self):
        return self._opts


class Classifier(nn.Module):
    def __init__(self, opts=None):
        super().__init__()
        if opts is None:
            opts = {"input_channels": 512, "pooling": "avg", "num_classes": 2}
        self._opts = opts
        self._layer_output = dict()
        if opts["pooling"] == "avg":
            self.pooling = lambda x: torch.mean(x, dim=-1)
        elif opts["pooling"] == "max":
            self.pooling = lambda x: torch.max(x, dim=-1)[0]
        else:
            raise ValueError("Unkown pooling option")
        self.linear = nn.Linear(opts["input_channels"], opts["num_classes"])

    def forward(self, x):
        x = x.view(x.size(0), x.size(1), -1)
        hidden_layer = self.pooling(x)
        hidden_layer = hidden_layer.view(hidden_layer.size(0), -1)
        self._layer_output["hidden_layer_1"] = hidden_layer
        output_layer = self.linear(hidden_layer)
        self._layer_output["output_layer"] = output_layer
        return output_layer

    def model_opts(self):
        return self._opts

def _assemble(encoder, classifier):
    model = nn.Sequential(OrderedDict([('encoder', encoder), ('classifier', classifier)]))
    model.eval()
    return model


def build_model(encoderOpts, num_classes, device):
    """Build a fresh model (random init) with a num_classes-wide head."""
    classifierOpts = {'input_channels': 512, 'pooling': 'avg', 'num_classes': num_classes}
    encoder = ResidualEncoder(encoderOpts)
    classifier = Classifier(classifierOpts)
    return _assemble(encoder, classifier), encoder, classifier, classifierOpts


def load_model_from_pk(path, device):
    """Load a .pk file. Returns (model, classes_dict, dataOpts)."""
    obj = torch.load(path, map_location=device, weights_only=False)
    encoder = ResidualEncoder(obj['encoderOpts'])
    encoder.load_state_dict(obj['encoderState'])
    classifier = Classifier(obj['classifierOpts'])
    classifier.load_state_dict(obj['classifierState'])
    return _assemble(encoder, classifier), obj.get('classes', {}), obj.get('dataOpts', {})


# .checkpoint files (Data/model_output/*) store no opts, so these are supplied
# from that training run's config. Verify against your own TRAIN.log if in doubt.
CHECKPOINT_ENCODER_OPTS = {'input_channels': 1, 'conv_kernel_size': 7,
                           'max_pool': 2, 'resnet_size': 18}
CHECKPOINT_DATA_OPTS = {'sr': 384000, 'preemphases': 0.98, 'n_fft': 256,
                        'hop_length': 128, 'n_freq_bins': 256, 'fmin': 18000,
                        'fmax': 90000, 'freq_compression': 'linear',
                        'min_level_db': -100, 'ref_level_db': 20}


def load_any_model(path, device):
    """Load either a .pk or a .checkpoint file."""
    obj = torch.load(path, map_location=device, weights_only=False)
    if 'encoderOpts' in obj:
        return load_model_from_pk(path, device)
    if 'modelState' in obj:
        classes = obj.get('classes', {})
        classifierOpts = {'input_channels': 512, 'pooling': 'avg',
                          'num_classes': len(classes)}
        encoder = ResidualEncoder(CHECKPOINT_ENCODER_OPTS)
        classifier = Classifier(classifierOpts)
        ms = obj['modelState']
        enc_state = {k[len('encoder.'):]: v for k, v in ms.items() if k.startswith('encoder.')}
        cls_state = {k[len('classifier.'):]: v for k, v in ms.items() if k.startswith('classifier.')}
        encoder.load_state_dict(enc_state)
        classifier.load_state_dict(cls_state)
        return _assemble(encoder, classifier), classes, dict(CHECKPOINT_DATA_OPTS)
    raise ValueError(f'Unknown model format: {list(obj.keys())}')


print('Architecture + loading functions defined')


In [ ]:
# Cell 6: Transforms & Dataset

def load_audio_file(path, target_sr):
    """Load audio and resample to target_sr."""
    audio, native_sr = sf.read(path, dtype='float32')
    if audio.ndim > 1:
        audio = audio[:, 0]
    if native_sr != target_sr:
        audio = resampy.resample(audio, native_sr, target_sr)
    return audio, target_sr

def pre_emphasize(audio, factor=0.98):
    """Apply pre-emphasis filter."""
    return np.append(audio[0], audio[1:] - factor * audio[:-1])

def compute_spectrogram(audio, n_fft, hop_length):
    """Compute STFT spectrogram."""
    window = np.hanning(n_fft)
    n_frames = 1 + (len(audio) - n_fft) // hop_length
    frames = np.stack([audio[i*hop_length : i*hop_length+n_fft] for i in range(n_frames)])
    spec = np.abs(np.fft.rfft(frames * window, n=n_fft))**2
    return spec

def freq_compress(spec, compression, n_freq_bins, fmin, fmax, sr):
    """Apply frequency compression (linear or mel)."""
    n_fft_bins = spec.shape[1]
    freqs = np.linspace(0, sr/2, n_fft_bins)
    mask = (freqs >= fmin) & (freqs <= fmax)
    spec = spec[:, mask]
    if compression == 'linear':
        # Interpolate to n_freq_bins
        from scipy.ndimage import zoom
        zoom_factor = n_freq_bins / spec.shape[1]
        spec = zoom(spec, (1, zoom_factor))
    return spec

def amp_to_db(spec, min_level_db=-100):
    """Convert amplitude to dB scale."""
    spec = 10 * np.log10(np.maximum(spec, 1e-10))
    spec = np.maximum(spec, min_level_db)
    return spec

def normalize(spec):
    """Normalize spectrogram to [0, 1]."""
    spec = spec - spec.min()
    spec_max = spec.max()
    if spec_max > 0:
        spec = spec / spec_max
    return spec

def get_class_from_filename(filename):
    """Extract class label from BatSpot filename (before first '-')."""
    return os.path.basename(filename).split('-', 1)[0]


class BatDataset(Dataset):
    """Dataset that loads wav files and computes spectrograms on-the-fly (slow; kept for fallback)."""

    def __init__(self, file_names, class_to_idx, sr, n_fft, hop_length,
                 n_freq_bins, f_min, f_max, seq_len, freq_compression='linear'):
        self.file_names = file_names
        self.class_to_idx = class_to_idx
        self.sr = sr
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.n_freq_bins = n_freq_bins
        self.f_min = f_min
        self.f_max = f_max
        self.seq_len = seq_len
        self.freq_compression = freq_compression

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        path = self.file_names[idx]
        audio, _ = load_audio_file(path, self.sr)
        audio = pre_emphasize(audio)
        spec = compute_spectrogram(audio, self.n_fft, self.hop_length)
        spec = freq_compress(spec, self.freq_compression, self.n_freq_bins, self.f_min, self.f_max, self.sr)
        spec = amp_to_db(spec)
        spec = normalize(spec)
        # Truncate or pad to seq_len
        if spec.shape[0] >= self.seq_len:
            spec = spec[:self.seq_len]
        else:
            pad = np.zeros((self.seq_len - spec.shape[0], spec.shape[1]))
            spec = np.concatenate([spec, pad], axis=0)
        spec = torch.FloatTensor(spec).unsqueeze(0)  # (1, seq_len, n_freq_bins)
        label = self.class_to_idx[get_class_from_filename(path)]
        return spec, label


def _augment_spec(spec):
    """Apply on-the-fly spectrogram augmentation (training only).
    Returns a new numpy array; input shape (seq_len, n_freq_bins).
    Techniques: time shift, Gaussian noise, SpecAugment freq & time masking.
    """
    seq_len, n_freq = spec.shape
    spec = spec.copy()

    # Time shift: ±20 % of seq_len (zero-fill wrapped edge)
    shift = int(np.random.uniform(-0.20, 0.20) * seq_len)
    if shift > 0:
        spec = np.concatenate([np.zeros((shift, n_freq)), spec[:-shift]], axis=0)
    elif shift < 0:
        spec = np.concatenate([spec[-shift:], np.zeros((-shift, n_freq))], axis=0)

    # Gaussian noise
    spec = spec + np.random.normal(0, 0.01, spec.shape)
    spec = np.clip(spec, 0.0, 1.0)

    # SpecAugment — frequency masking (up to 20 % of freq bins)
    f_mask = int(np.random.uniform(0, 0.20) * n_freq)
    if f_mask > 0:
        f0 = np.random.randint(0, max(1, n_freq - f_mask))
        spec[:, f0:f0 + f_mask] = 0.0

    # SpecAugment — time masking (up to 20 % of frames)
    t_mask = int(np.random.uniform(0, 0.20) * seq_len)
    if t_mask > 0:
        t0 = np.random.randint(0, max(1, seq_len - t_mask))
        spec[t0:t0 + t_mask, :] = 0.0

    return spec


class CachedBatDataset(Dataset):
    """Fast dataset: spectrograms pre-computed to disk on first access.

    Eliminates the per-sample resampy + STFT + zoom CPU bottleneck that starved
    GPUs. After the first epoch the cache is served by np.load (~microseconds).
    Augmentation is applied on-the-fly AFTER loading from cache (training only).
    """
    _CACHE_ROOT = '/kaggle/working/spec_cache'

    def __init__(self, file_names, class_to_idx, sr, n_fft, hop_length,
                 n_freq_bins, f_min, f_max, seq_len, freq_compression='linear',
                 tag='split', augment=False):
        self.file_names = file_names
        self.class_to_idx = class_to_idx
        self.sr = sr
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.n_freq_bins = n_freq_bins
        self.f_min = f_min
        self.f_max = f_max
        self.seq_len = seq_len
        self.freq_compression = freq_compression
        self.augment = augment
        _cfg_key = f"{sr}_{n_fft}_{hop_length}_{n_freq_bins}_{f_min}_{f_max}_{seq_len}_{freq_compression}"
        _cfg_hash = __import__("hashlib").md5(_cfg_key.encode()).hexdigest()[:8]
        self.cache_dir = os.path.join(self._CACHE_ROOT, f"{tag}_{_cfg_hash}")
        os.makedirs(self.cache_dir, exist_ok=True)
        self._build_cache()

    def _spec_path(self, idx):
        fname = os.path.basename(self.file_names[idx])
        return os.path.join(self.cache_dir, fname + '.npy')

    def _build_cache(self):
        missing = [i for i in range(len(self.file_names))
                   if not os.path.exists(self._spec_path(i))]
        if not missing:
            return
        print(f'[CachedBatDataset] Building cache for {len(missing)} files '
              f'({self.cache_dir})...')
        for i in tqdm(missing, desc='cache', leave=False):
            path = self.file_names[i]
            audio, _ = load_audio_file(path, self.sr)
            audio = pre_emphasize(audio)
            spec = compute_spectrogram(audio, self.n_fft, self.hop_length)
            spec = freq_compress(spec, self.freq_compression, self.n_freq_bins,
                                 self.f_min, self.f_max, self.sr)
            spec = amp_to_db(spec)
            spec = normalize(spec)
            if spec.shape[0] >= self.seq_len:
                spec = spec[:self.seq_len]
            else:
                pad = np.zeros((self.seq_len - spec.shape[0], spec.shape[1]))
                spec = np.concatenate([spec, pad], axis=0)
            np.save(self._spec_path(i), spec.astype(np.float32))

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        spec = np.load(self._spec_path(idx))   # (seq_len, n_freq_bins)
        if self.augment:
            spec = _augment_spec(spec)
        spec = torch.FloatTensor(spec).unsqueeze(0)  # (1, seq_len, n_freq_bins)
        label = self.class_to_idx[get_class_from_filename(self.file_names[idx])]
        return spec, label


print('Transforms + BatDataset + CachedBatDataset defined')


In [ ]:
# Cell 7: Data Discovery

# Find all wav files
all_wavs = sorted(glob.glob(os.path.join(DATA_DIR, '**', '*.wav'), recursive=True))
if not all_wavs:
    # Try flat structure
    all_wavs = sorted(glob.glob(os.path.join(DATA_DIR, '*.wav')))

print(f'Found {len(all_wavs)} wav files')

# Auto-detect classes from folder names
class_counts = Counter()
for w in all_wavs:
    cls = get_class_from_filename(w)
    class_counts[cls] += 1

all_classes = sorted(class_counts.keys())
print(f'Classes ({len(all_classes)}):')
for c in all_classes:
    print(f'  {c}: {class_counts[c]} files')

# Update classifier config with actual num_classes
CLS_CONFIG['num_classes'] = len(all_classes)
print(f'\nClassifier num_classes: {CLS_CONFIG["num_classes"]}')

# Detector class mapping: noise -> 0, all bats -> 1
DET_CLASSES = ['noise', 'call']
DET_CLASS_TO_IDX = {'noise': 0, 'call': 1}

# Classifier class mapping: sorted classes
CLS_CLASSES = all_classes
CLS_CLASS_TO_IDX = {c: i for i, c in enumerate(all_classes)}
CLS_IDX_TO_CLASS = {i: c for c, i in CLS_CLASS_TO_IDX.items()}

print(f'\nDetector mapping: {DET_CLASS_TO_IDX}')
print(f'Classifier mapping: {CLS_CLASS_TO_IDX}')

# Split data: 70/15/15 train/val/test (stratified)
all_labels = [get_class_from_filename(w) for w in all_wavs]
train_wavs, test_wavs, train_labels, test_labels = train_test_split(
    all_wavs, all_labels, test_size=0.15, random_state=42, stratify=all_labels)
train_wavs, val_wavs, train_labels, val_labels = train_test_split(
    train_wavs, train_labels, test_size=0.15/0.85, random_state=42, stratify=train_labels)

print(f'\nSplit: train={len(train_wavs)}, val={len(val_wavs)}, test={len(test_wavs)}')
print('Train class dist:', Counter(train_labels))
print('Val class dist:', Counter(val_labels))
print('Test class dist:', Counter(test_labels))

In [ ]:
# Cell 8: Clip Extraction (Python port of export_clips.R)

def extract_clips_from_selections(raw_audio_dir, selections_dir, output_dir, target_sr=384000):
    """
    Extract audio clips from raw recordings using Raven selection tables.
    Python port of Data/scripts/export_clips.R.
    """
    os.makedirs(output_dir, exist_ok=True)
    extracted = 0

    # Iterate over species folders in selections_dir
    for species in sorted(os.listdir(selections_dir)):
        species_dir = os.path.join(selections_dir, species)
        if not os.path.isdir(species_dir):
            continue

        for sel_file in sorted(glob.glob(os.path.join(species_dir, '*.txt'))):
            # Read Raven selection table (tab-separated)
            with open(sel_file, 'r') as f:
                lines = f.readlines()
            if len(lines) < 2:
                continue

            # Parse header
            header = lines[0].strip().split('\t')
            try:
                begin_idx = header.index('Begin Time (s)')
                end_idx = header.index('End Time (s)')
                view_idx = header.index('View')
            except ValueError:
                continue

            # Find matching raw audio file by date/time in selection table filename
            # Selection filename format: YYYYMMDD_HHMMSS_species.txt
            sel_name = os.path.basename(sel_file).replace('.txt', '')
            date_time = '_'.join(sel_name.split('_')[:2])
            matching_audio = None
            for audio_file in glob.glob(os.path.join(raw_audio_dir, '*.WAV')):
                audio_name = os.path.basename(audio_file).replace('.WAV', '')
                if date_time in audio_name:
                    matching_audio = audio_file
                    break
            if matching_audio is None:
                continue

            # Load audio
            audio, native_sr = sf.read(matching_audio, dtype='float32')
            if audio.ndim > 1:
                audio = audio[:, 0]

            # Extract each selection
            for line in lines[1:]:
                parts = line.strip().split('\t')
                if len(parts) <= max(begin_idx, end_idx, view_idx):
                    continue
                view = parts[view_idx]
                if 'Spectrogram' not in view:
                    continue
                begin_time = float(parts[begin_idx])
                end_time = float(parts[end_idx])
                start_sample = int(begin_time * native_sr)
                end_sample = int(end_time * native_sr)
                clip = audio[start_sample:end_sample]
                if len(clip) == 0:
                    continue
                start_ms = int(begin_time * 1000)
                end_ms = int(end_time * 1000)
                out_name = f'{species}-bat_{date_time}_{sel_name}_{start_ms}_{end_ms}.wav'
                out_path = os.path.join(output_dir, out_name)
                sf.write(out_path, clip, native_sr)
                extracted += 1

    print(f'Extracted {extracted} clips to {output_dir}')
    return extracted

# Extraction is opt-in, and refuses to clobber an existing dataset.
if not RUN_CLIP_EXTRACTION:
    print('Clip extraction SKIPPED (RUN_CLIP_EXTRACTION = False in Cell 2).')
    print(f'Using the clips already present in: {DATA_DIR}')
elif not (RAW_AUDIO_DIR and SELECTIONS_DIR):
    print('Clip extraction skipped: set RAW_AUDIO_DIR and SELECTIONS_DIR in Cell 2.')
else:
    _existing = glob.glob(os.path.join(DATA_DIR, '**', '*.wav'), recursive=True)
    if _existing and not OVERWRITE_EXISTING_CLIPS:
        print(f'Clip extraction ABORTED: {len(_existing)} wav files already exist in')
        print(f'  {DATA_DIR}')
        print('Re-extracting would overwrite them. Set OVERWRITE_EXISTING_CLIPS = True')
        print('in Cell 2 to force it, or point DATA_DIR at an empty folder.')
    else:
        if _existing:
            print(f'WARNING: overwriting {len(_existing)} existing clips in {DATA_DIR}')
        extract_clips_from_selections(RAW_AUDIO_DIR, SELECTIONS_DIR, DATA_DIR)

In [ ]:
# Cell 9: Training Infrastructure

def compute_class_weights(file_names, class_to_idx, num_labels=None):
    """Inverse-frequency class weights, one per MODEL OUTPUT class.

    num_labels must be the classifier head size (config['num_classes']), not
    len(class_to_idx): the detector folds 8 bat species + 'noise' into 2
    labels, so keying the vector off the dict length yields a 9-element
    tensor for a 2-output head and CrossEntropyLoss rejects it.
    """
    labels = [class_to_idx[get_class_from_filename(f)] for f in file_names]
    counts = Counter(labels)
    total = len(labels)
    n = num_labels if num_labels else len(class_to_idx)
    weights = torch.zeros(n)
    n_present = len(counts)
    for idx, count in counts.items():
        weights[idx] = total / (n_present * count) if count else 0.0
    return weights

def make_weighted_sampler(file_names, class_to_idx, num_labels=None):
    """Create WeightedRandomSampler for class balancing."""
    labels = [class_to_idx[get_class_from_filename(f)] for f in file_names]
    weights = compute_class_weights(file_names, class_to_idx, num_labels)
    sample_weights = [weights[l] for l in labels]
    return WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

def train_model(model, train_dataset, val_dataset, config, device, save_path=None):
    """
    Train model with AMP, gradient accumulation, DataParallel.
    Returns (model, history, best_val_acc).
    """
    # Compute class weights
    all_file_names = train_dataset.file_names
    class_to_idx = train_dataset.class_to_idx
    class_weights = compute_class_weights(
        all_file_names, class_to_idx, config.get('num_classes')).to(device)
    print(f'Class weights: {class_weights.tolist()}')

    # Weighted sampler for training
    sampler = make_weighted_sampler(all_file_names, class_to_idx,
                                   config.get('num_classes'))

    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'],
                              sampler=sampler, num_workers=4, pin_memory=True,
                              persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'],
                            shuffle=False, num_workers=4, pin_memory=True,
                            persistent_workers=True)

    # Loss, optimizer, scheduler
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    effective_lr = config['base_lr']  # paper LR is absolute; do NOT scale by batch size
    optimizer = optim.Adam(model.parameters(), lr=effective_lr, betas=(0.5, 0.999))
    patience_lr = int(max(1, ceil(8 / config['epochs_per_eval'])))
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=patience_lr, factor=0.5, threshold=1e-3, threshold_mode='abs')
    patience_early = int(max(1, ceil(20 / config['epochs_per_eval'])))
    grad_accum_steps = config.get('grad_accum_steps', 1)
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    # Multi-GPU
    is_dp = config.get('use_multi_gpu', False) and torch.cuda.device_count() > 1
    if is_dp:
        model = nn.DataParallel(model)
        print(f'DataParallel across {torch.cuda.device_count()} GPUs')

    print(f'Grad accum: {grad_accum_steps}, AMP: {use_amp}, DP: {is_dp}')
    print(f'Effective LR: {effective_lr:.2e}')
    print(f'Patience LR: {patience_lr}, Patience ES: {patience_early}')

    history = {'train_loss': [], 'val_acc': []}
    best_acc = 0.0
    best_state = None
    no_improve = 0

    for epoch in range(config['n_epochs']):
        # Train
        model.train()
        epoch_loss = 0.0
        n_batches = 0
        optimizer.zero_grad()

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config["n_epochs"]}', leave=False)
        for step, (inputs, labels) in enumerate(pbar):
            inputs, labels = inputs.to(device), labels.to(device)
            with torch.amp.autocast(device.type, enabled=use_amp):
                output = model(inputs)
                loss = loss_fn(output, labels) / grad_accum_steps
            scaler.scale(loss).backward()
            if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            epoch_loss += loss.item() * grad_accum_steps
            n_batches += 1

        avg_loss = epoch_loss / n_batches
        history['train_loss'].append(avg_loss)

        # Validate every epochs_per_eval
        if (epoch + 1) % config['epochs_per_eval'] == 0 or epoch == config['n_epochs'] - 1:
            model.eval()
            all_preds, all_labels = [], []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs = inputs.to(device)
                    output = model(inputs)
                    preds = output.argmax(dim=1).cpu().numpy()
                    all_preds.extend(preds)
                    all_labels.extend(labels.numpy())
            val_acc = accuracy_score(all_labels, all_preds)
            history['val_acc'].append(val_acc)
            scheduler.step(val_acc)

            if val_acc > best_acc:
                best_acc = val_acc
                no_improve = 0
                if save_path:
                    # Always save bare keys (strip DataParallel's "module." prefix)
                    _sd = model.module.state_dict() if is_dp else model.state_dict()
                    best_state = {k: v.clone() for k, v in _sd.items()}
            else:
                no_improve += 1

            print(f'  Epoch {epoch+1}: loss={avg_loss:.4f} val_acc={val_acc:.4f} best={best_acc:.4f} lr={optimizer.param_groups[0]["lr"]:.2e}')

            if no_improve >= patience_early:
                print(f'Early stopping at epoch {epoch+1}.')
                break

    # Restore best weights (best_state always has bare keys)
    if save_path and best_state is not None:
        if is_dp:
            model.module.load_state_dict(best_state)
        else:
            model.load_state_dict(best_state)

    return model, history, best_acc

print('Training infrastructure defined')


In [ ]:
# Cell 10: Load Pre-trained Models from the uploaded Kaggle dataset
#
# Expects the official BatSpot models (extracted from BatSpot_article):
#   <dataset>/batspot/models_call_detector/m03|m09|m11/train/ANIMAL-SPOT.pk
#   <dataset>/batspot/models_call_classifier/m09/train/ANIMAL-SPOT.pk
#
# NOTE: all four files share the basename ANIMAL-SPOT.pk, so each is staged
# here under a UNIQUE name. Copying by basename would let the classifier
# overwrite the detector and silently fine-tune the detector from the
# 15-class classifier weights.

os.makedirs(PRETRAINED_DIR, exist_ok=True)

# --- Priority 1: explicit paths from Cell 2 ---
_official = {}
_manual_set = set()
_missing_manual = []
_manual_set_attempted = False
if USE_MANUAL_MODEL_PATHS:
    for _lbl, _role, _mic, _p in [
            ('detector m03', 'detector', 'm03', DETECTOR_M03_PATH),
            ('detector m09', 'detector', 'm09', DETECTOR_M09_PATH),
            ('detector m11', 'detector', 'm11', DETECTOR_M11_PATH),
            ('classifier m09', 'classifier', 'm09', CLASSIFIER_M09_PATH)]:
        if _p is None:
            continue
        _manual_set_attempted = True
        if os.path.isfile(_p):
            _ap = os.path.abspath(_p)
            _manual_set.add(_ap)
            _official.setdefault(_role, {})[_mic] = _ap
        else:
            _missing_manual.append(f'{_lbl}: {_p}')
    if _official:
        print(f'Using {sum(len(v) for v in _official.values())} manual path(s) from Cell 2.')
    for _m in _missing_manual:
        print(f'  manual path NOT FOUND -> {_m}')
    if _missing_manual:
        print('  falling back to auto-discovery for the missing ones.')
else:
    print('USE_MANUAL_MODEL_PATHS = False -> ignoring Cell 2 paths.')

# --- Priority 2: discover the dataset directory (local repo or /kaggle/input/*) ---
BATSPOT_ARTICLE_DIR = None
_MODEL_GLOB = os.path.join('models_*', '*', 'train', 'ANIMAL-SPOT.pk')

for _cand in ['BatSpot_article/batspot', 'BatSpot_article']:
    if glob.glob(os.path.join(_cand, _MODEL_GLOB)):
        BATSPOT_ARTICLE_DIR = _cand
        break

if BATSPOT_ARTICLE_DIR is None:
    for _root in sorted(glob.glob('/kaggle/input/*')):
        for _cand in [os.path.join(_root, 'batspot'), _root]:
            if glob.glob(os.path.join(_cand, _MODEL_GLOB)):
                BATSPOT_ARTICLE_DIR = _cand
                break
        if BATSPOT_ARTICLE_DIR:
            break

# --- Collect sources, keyed role -> mic -> path ---
#
# _official ALREADY holds whatever the manual block above resolved. Do NOT
# re-initialise it here: doing so discards the Cell 2 paths and makes this
# cell report "No pre-trained models found" even though the manual lookup
# just succeeded. Discovery only fills roles/mics the manual block missed.
_manual_n = sum(len(v) for v in _official.values())
_skipped = []
if BATSPOT_ARTICLE_DIR:
    for _p in sorted(glob.glob(os.path.join(BATSPOT_ARTICLE_DIR, _MODEL_GLOB))):
        _parts = os.path.relpath(_p, BATSPOT_ARTICLE_DIR).split(os.sep)
        # e.g. ['models_call_detector', 'm09', 'train', 'ANIMAL-SPOT.pk']
        if len(_parts) < 4:
            continue
        _family, _mic = _parts[0], _parts[1]
        # Resolve the role FIRST, and only ever to 'classifier' or 'detector'.
        # buzz_detector / social_detector use different band limits and are
        # not fine-tuned here; classifying them as detectors would feed their
        # weights into the call-detector slot.
        if _family == 'models_call_classifier':
            _role = 'classifier'
        elif _family == 'models_call_detector':
            _role = 'detector'
        else:
            _skipped.append(f'{_family}/{_mic}')
            continue
        if _mic in _official.get(_role, {}):
            continue  # already supplied manually via Cell 2 (manual wins)
        _official.setdefault(_role, {})[_mic] = _p
    if _manual_n:
        print(f'Auto-discovery added nothing (Cell 2 supplied {_manual_n} model(s)); '
              f'manual paths stand.')

# --- Stage under unique names and read REAL metadata from each .pk ---
# PRETRAINED_MODELS[role][mic] = {path, sr, classes, num_classes, data, source}
PRETRAINED_MODELS = {}
detector_path = None
classifier_path = None

if _official:
    if BATSPOT_ARTICLE_DIR:
        print(f'BatSpot_article dataset: {BATSPOT_ARTICLE_DIR}\n')
    for _role, _mics in _official.items():
        for _mic, _src in sorted(_mics.items()):
            _dest = os.path.join(PRETRAINED_DIR, f'official_{_role}_{_mic}.pk')
            shutil.copy2(_src, _dest)
            try:
                _o = torch.load(_dest, map_location='cpu', weights_only=False)
                _d = _o.get('dataOpts', {})
                PRETRAINED_MODELS.setdefault(_role, {})[_mic] = {
                    'path': _dest,
                    'data': _d,
                    'sr': _d.get('sr'),
                    'classes': _o.get('classes', {}),
                    'num_classes': _o.get('classifierOpts', {}).get('num_classes'),
                    'source': _src,
                    'origin': 'manual' if os.path.abspath(_src) in _manual_set else 'discovered',
                }
            except Exception as e:
                print(f'  WARNING: could not read {_dest}: {e}')
    print('Official models staged:')
    for _role in sorted(PRETRAINED_MODELS):
        for _mic, _info in sorted(PRETRAINED_MODELS[_role].items()):
            _sr = _info['sr'] or 0
            print(f'  {_role:9s} {_mic}: {_info["num_classes"]:2d} classes, {_sr//1000:3d} kHz'
                  f'  [{_info.get("origin", "?")}] <- {_info["source"]}')
    # m09 is the paper's primary microphone (call detector test F1 0.985).
    # Prefer it; fall back to the lowest-numbered variant.
    for _role in ('detector', 'classifier'):
        _mics = PRETRAINED_MODELS.get(_role, {})
        if not _mics:
            continue
        _pick = 'm09' if 'm09' in _mics else sorted(_mics)[0]
        if _role == 'detector':
            detector_path = _mics[_pick]['path']
        else:
            classifier_path = _mics[_pick]['path']
        print(f'  -> fine-tuning {_role} from {_pick}')
    if _skipped:
        print(f'  (skipped, not fine-tuned here: {sorted(set(_skipped))})')
elif not _official:
    print('No pre-trained models found.')
    if USE_MANUAL_MODEL_PATHS and _manual_set_attempted:
        print('All Cell 2 paths were set but none resolved to a readable file.')
        print('Re-run Cell 2 and check the per-path OK / NOT FOUND lines above it.')
    else:
        print('Either set DETECTOR_M03_PATH / DETECTOR_M09_PATH / DETECTOR_M11_PATH /')
        print('CLASSIFIER_M09_PATH in Cell 2, or attach the dataset via Add Data.')

# --- Verify our spectrogram config matches each official model's dataOpts ---
print('\nConfig vs model dataOpts check:')
for _role, _cfg in (('detector', DET_CONFIG), ('classifier', CLS_CONFIG)):
    _mics = PRETRAINED_MODELS.get(_role)
    if not _mics:
        continue
    for _mic, _info in sorted(_mics.items()):
        _d = _info['data']
        _bad = []
        for _k in ('sr', 'n_fft', 'hop_length', 'fmin', 'fmax', 'n_freq_bins', 'freq_compression'):
            if _k in _d and _k in _cfg and _d[_k] != _cfg[_k]:
                _bad.append(f'{_k}: model={_d[_k]} config={_cfg[_k]}')
        _tag = f'official {_role} ({_mic})'
        if _bad:
            print(f'  MISMATCH {_tag}: ' + '; '.join(_bad))
        else:
            print(f'  OK        {_tag}')

# --- Class-overlap report (official models use European species codes) ---
if 'classes' in dir() or 'classes' in globals():
    _mine = list(globals().get('classes', []))
    for _role, _cfg in (('detector', DET_CONFIG), ('classifier', CLS_CONFIG)):
        for _mic, _info in sorted(PRETRAINED_MODELS.get(_role, {}).items()):
            _theirs = set(_info['classes'].keys())
            _ov = sorted(set(_mine) & _theirs)
            print(f'\n{_role} ({_mic}) knows {len(_theirs)} classes; '
                  f'overlap with your data {sorted(_mine)}: {_ov or "NONE"}')
            if not _ov and _role == 'classifier':
                print('  -> classifier head cannot transfer; only the encoder will be used.')
            elif _ov and _role == 'detector':
                print('  -> evaluated on the overlapping classes only (noise).')

# --- Fallback: local Data/model_output (384 kHz, 3-class) ---
if detector_path is None and classifier_path is None:
    _local = None
    for _cand in ['Data/model_output/ANIMAL-SPOT.pk',
                  '/kaggle/input/batspot-data/model_output/ANIMAL-SPOT.pk']:
        if os.path.isfile(_cand):
            _local = _cand
            break
    if _local is None and MODEL_OUTPUT_DIR and os.path.isdir(MODEL_OUTPUT_DIR):
        for _root, _, _files in os.walk(MODEL_OUTPUT_DIR):
            for _f in _files:
                if _f.endswith('.pk'):
                    _local = os.path.join(_root, _f)
                    break
            if _local:
                break
    if _local:
        _dest = os.path.join(PRETRAINED_DIR, 'fallback_local.pk')
        shutil.copy2(_local, _dest)
        detector_path = _dest
        classifier_path = _dest
        print(f'\nFallback local model: {_local}')
        print('  NOTE: 384 kHz / 3-class model. Spectrogram params will NOT match')
        print('        DET_CONFIG or CLS_CONFIG, so encoder transfer is unreliable.')
    else:
        print('\nWARNING: no pre-trained model found. Both models train from scratch.')

print(f'\nFine-tuning starting points:')
print(f'  Detector:   {detector_path}')
print(f'  Classifier: {classifier_path}')

In [ ]:
# Cell 11: Build Detector Dataset (shared across all 3 detector variants)

# Detector labels are binary: noise -> 0, EVERY bat species -> 1.
#
# class_to_idx is a plain lookup consumed by CachedBatDataset.__getitem__ via
# get_class_from_filename(), which returns the species from the filename
# (e.g. 'rhro'). A 2-entry dict {'noise': 0, 'call': 1} therefore raised
# KeyError on the first species file, so build the lookup over every class
# actually present in the data and fold all non-noise species to label 1.
_det_all = list(train_wavs) + list(val_wavs) + list(test_wavs)
_det_species = sorted({get_class_from_filename(f) for f in _det_all})
if 'noise' not in _det_species:
    print("WARNING: no 'noise' files in the data; the detector sees only calls.")

det_class_to_idx = {c: (0 if c == 'noise' else 1) for c in _det_species}
det_class_to_idx.setdefault('noise', 0)   # keep canonical keys even if absent
det_class_to_idx.setdefault('call', 1)
det_idx_to_class = {0: 'noise', 1: 'call'}

print(f'Detector: {len(_det_species)} dataset classes -> 2 labels')
print(f'  label 0 (noise): {[c for c, v in det_class_to_idx.items() if v == 0]}')
print(f'  label 1 (call):  {[c for c, v in det_class_to_idx.items() if v == 1]}')

# Compute seq_len for detector
det_seq_len = int(float(DET_CONFIG['sequence_len']) / 1000 * DET_CONFIG['sr'] / DET_CONFIG['hop_length'])
print(f'Detector: sr={DET_CONFIG["sr"]}, seq_len={det_seq_len} frames')

det_train = CachedBatDataset(train_wavs, det_class_to_idx, DET_CONFIG['sr'],
                              DET_CONFIG['n_fft'], DET_CONFIG['hop_length'],
                              DET_CONFIG['n_freq_bins'], DET_CONFIG['fmin'], DET_CONFIG['fmax'],
                              det_seq_len, DET_CONFIG['freq_compression'], tag='det_train',
                              augment=USE_AUGMENTATION)
det_val   = CachedBatDataset(val_wavs, det_class_to_idx, DET_CONFIG['sr'],
                              DET_CONFIG['n_fft'], DET_CONFIG['hop_length'],
                              DET_CONFIG['n_freq_bins'], DET_CONFIG['fmin'], DET_CONFIG['fmax'],
                              det_seq_len, DET_CONFIG['freq_compression'], tag='det_val')
det_test  = CachedBatDataset(test_wavs, det_class_to_idx, DET_CONFIG['sr'],
                              DET_CONFIG['n_fft'], DET_CONFIG['hop_length'],
                              DET_CONFIG['n_freq_bins'], DET_CONFIG['fmin'], DET_CONFIG['fmax'],
                              det_seq_len, DET_CONFIG['freq_compression'], tag='det_test')

# Show class distribution for detector
det_train_labels = [det_class_to_idx[get_class_from_filename(f)] for f in train_wavs]
det_val_labels   = [det_class_to_idx[get_class_from_filename(f)] for f in val_wavs]
det_test_labels  = [det_class_to_idx[get_class_from_filename(f)] for f in test_wavs]
print(f'Detector train: {Counter(det_train_labels)}')
print(f'Detector val:   {Counter(det_val_labels)}')
print(f'Detector test:  {Counter(det_test_labels)}')


In [ ]:
# Cell 12: Train ALL 3 detector variants (m03, m09, m11) + evaluate each individually.
#
# PRETRAINED_MODELS['detector'] = {'m03': {...}, 'm09': {...}, 'm11': {...}}
# For each mic: load its pretrained encoder, fine-tune, evaluate on det_test.
# Results stored in det_results[mic].

def evaluate_model(model, dataset, class_names, device, model_name='Model'):
    """Evaluate model on dataset. Returns metrics dict."""
    model.eval()
    loader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=4,
                        persistent_workers=True)
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            output = model(inputs)
            probs = torch.softmax(output, dim=1)
            preds = output.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)
    eval_labels = list(range(len(class_names)))
    acc  = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='macro', zero_division=0)
    report = classification_report(all_labels, all_preds, labels=eval_labels,
                                   target_names=class_names, zero_division=0,
                                   output_dict=True)
    cm = confusion_matrix(all_labels, all_preds, labels=eval_labels)
    print(f'\n=== {model_name} ===')
    print(f'Accuracy:  {acc:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}')
    print(classification_report(all_labels, all_preds, labels=eval_labels,
                                target_names=class_names, zero_division=0))
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
            'predictions': all_preds, 'labels': all_labels, 'probs': all_probs,
            'confusion_matrix': cm, 'report': report, 'class_names': class_names}


encoderOpts_det = {'input_channels': 1, 'conv_kernel_size': 7, 'max_pool': 2, 'resnet_size': 18}
_det_mics = sorted(PRETRAINED_MODELS.get('detector', {}).keys())
if not _det_mics:
    print('No pretrained detector variants found -> training one from scratch.')
    _det_mics = ['scratch']

# det_results[mic] = {'model', 'metrics', 'best_val_acc', 'encoderOpts', 'classifierOpts'}
det_results = {}

for _mic in _det_mics:
    print(f'\n{"="*60}')
    print(f'DETECTOR VARIANT: {_mic}')
    print(f'{"="*60}')

    model, encoder, _cls_head, classifierOpts = build_model(
        encoderOpts_det, DET_CONFIG['num_classes'], DEVICE)

    _pk = PRETRAINED_MODELS.get('detector', {}).get(_mic, {}).get('path')
    if _pk:
        try:
            _pre, _, _pre_opts = load_model_from_pk(_pk, DEVICE)
            encoder.load_state_dict(dict(list(_pre.named_children())[0][1].state_dict()))
            print(f'Encoder loaded from {os.path.basename(_pk)} ({_pre_opts.get("sr", 0)//1000}kHz)')
        except Exception as _e:
            print(f'Could not load encoder for {_mic}: {_e}')
    else:
        print(f'No pretrained path for {_mic} -> random init.')

    model = model.to(DEVICE)
    print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

    model, _hist, best_acc = train_model(
        model, det_train, det_val, DET_CONFIG, DEVICE,
        save_path=os.path.join(WORKING_DIR, f'detector_{_mic}_curves.png'),
    )
    print(f'Best val accuracy ({_mic}): {best_acc:.4f}')

    metrics = evaluate_model(model, det_test, DET_CLASSES, DEVICE,
                             f'Detector {_mic} (Test Set)')

    # Confusion matrix
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(metrics['confusion_matrix'], interpolation='nearest', cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    ax.set(xticks=range(2), yticks=range(2),
           xticklabels=DET_CLASSES, yticklabels=DET_CLASSES,
           xlabel='Predicted', ylabel='True',
           title=f'Detector {_mic} Confusion Matrix (Test)')
    for i in range(2):
        for j in range(2):
            v = metrics['confusion_matrix'][i, j]
            ax.text(j, i, str(v), ha='center', va='center',
                    color='white' if v > metrics['confusion_matrix'].max()/2 else 'black')
    plt.tight_layout(); plt.show()

    det_results[_mic] = {'model': model, 'metrics': metrics,
                         'best_val_acc': best_acc,
                         'encoderOpts': encoderOpts_det,
                         'classifierOpts': classifierOpts}

# Comparison table
print(f'\n{"="*60}')
print('DETECTOR COMPARISON SUMMARY')
print(f'{"="*60}')
print(f'{"Variant":<10} {"Val Acc":>8} {"Test Acc":>10} {"Prec":>8} {"Rec":>8} {"F1":>8}')
print('-'*60)
for _mic, _r in det_results.items():
    m = _r['metrics']
    print(f'{_mic:<10} {_r["best_val_acc"]:>8.4f} {m["accuracy"]:>10.4f} '
          f'{m["precision"]:>8.4f} {m["recall"]:>8.4f} {m["f1"]:>8.4f}')


In [ ]:
# Cell 13: Build Classifier Dataset + Train Classifier (single, shared across all pipelines)

cls_seq_len = int(float(CLS_CONFIG['sequence_len']) / 1000 * CLS_CONFIG['sr'] / CLS_CONFIG['hop_length'])
print(f'Classifier: sr={CLS_CONFIG["sr"]}, seq_len={cls_seq_len} frames')

cls_train = CachedBatDataset(train_wavs, CLS_CLASS_TO_IDX, CLS_CONFIG['sr'],
                              CLS_CONFIG['n_fft'], CLS_CONFIG['hop_length'],
                              CLS_CONFIG['n_freq_bins'], CLS_CONFIG['fmin'], CLS_CONFIG['fmax'],
                              cls_seq_len, CLS_CONFIG['freq_compression'], tag='cls_train',
                              augment=USE_AUGMENTATION)
cls_val   = CachedBatDataset(val_wavs, CLS_CLASS_TO_IDX, CLS_CONFIG['sr'],
                              CLS_CONFIG['n_fft'], CLS_CONFIG['hop_length'],
                              CLS_CONFIG['n_freq_bins'], CLS_CONFIG['fmin'], CLS_CONFIG['fmax'],
                              cls_seq_len, CLS_CONFIG['freq_compression'], tag='cls_val')
cls_test  = CachedBatDataset(test_wavs, CLS_CLASS_TO_IDX, CLS_CONFIG['sr'],
                              CLS_CONFIG['n_fft'], CLS_CONFIG['hop_length'],
                              CLS_CONFIG['n_freq_bins'], CLS_CONFIG['fmin'], CLS_CONFIG['fmax'],
                              cls_seq_len, CLS_CONFIG['freq_compression'], tag='cls_test')

print(f'Classifier train: {Counter([CLS_CLASS_TO_IDX[get_class_from_filename(f)] for f in train_wavs])}')
print(f'Classifier val:   {Counter([CLS_CLASS_TO_IDX[get_class_from_filename(f)] for f in val_wavs])}')
print(f'Classifier test:  {Counter([CLS_CLASS_TO_IDX[get_class_from_filename(f)] for f in test_wavs])}')

encoderOpts_cls = {'input_channels': 1, 'conv_kernel_size': 7, 'max_pool': 2, 'resnet_size': 18}
cls_model, cls_encoder, cls_classifier_head, cls_classifierOpts = build_model(
    encoderOpts_cls, CLS_CONFIG['num_classes'], DEVICE)

if classifier_path:
    try:
        _pre, _, _pre_opts = load_model_from_pk(classifier_path, DEVICE)
        cls_encoder.load_state_dict(dict(list(_pre.named_children())[0][1].state_dict()))
        print(f'Encoder loaded from {os.path.basename(classifier_path)} ({_pre_opts.get("sr",0)//1000}kHz)')
    except Exception as _e:
        print(f'Could not load classifier pretrained: {_e}')
else:
    print('No pre-trained classifier. Training from scratch.')

cls_model = cls_model.to(DEVICE)
print(f'Classifier params: {sum(p.numel() for p in cls_model.parameters()):,}')

cls_model, cls_history, cls_best_acc = train_model(
    cls_model, cls_train, cls_val, CLS_CONFIG, DEVICE,
    save_path=os.path.join(WORKING_DIR, 'classifier_curves.png'),
)
print(f'\nClassifier training complete. Best val accuracy: {cls_best_acc:.4f}')

cls_metrics = evaluate_model(cls_model, cls_test, CLS_CLASSES, DEVICE, 'Classifier (Test Set)')

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cls_metrics['confusion_matrix'], interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
n_cls = len(CLS_CLASSES)
ax.set(xticks=range(n_cls), yticks=range(n_cls),
       xticklabels=CLS_CLASSES, yticklabels=CLS_CLASSES,
       xlabel='Predicted', ylabel='True', title='Classifier Confusion Matrix (Test)')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
for i in range(n_cls):
    for j in range(n_cls):
        v = cls_metrics['confusion_matrix'][i, j]
        ax.text(j, i, str(v), ha='center', va='center',
                color='white' if v > cls_metrics['confusion_matrix'].max()/2 else 'black')
plt.tight_layout(); plt.show()


In [ ]:
# Cell 14: Combined Pipeline — each detector variant (m03/m09/m11) x the single classifier.
#
# For each mic in det_results:
#   1. Optimise detector threshold on val set (maximise call-class F1)
#   2. Detector -> Classifier on test set
#   3. Full metrics + confusion matrix
# Final summary table compares all variants.

def _spec_tensor(wav_path, cfg, seq_len):
    """Return (1,1,seq_len,n_freq_bins) float tensor for one wav file."""
    audio, _ = load_audio_file(wav_path, cfg['sr'])
    audio = pre_emphasize(audio)
    spec = compute_spectrogram(audio, cfg['n_fft'], cfg['hop_length'])
    spec = freq_compress(spec, cfg['freq_compression'], cfg['n_freq_bins'],
                         cfg['fmin'], cfg['fmax'], cfg['sr'])
    spec = amp_to_db(spec)
    spec = normalize(spec)
    if spec.shape[0] >= seq_len:
        spec = spec[:seq_len]
    else:
        spec = np.concatenate([spec, np.zeros((seq_len - spec.shape[0], spec.shape[1]))], axis=0)
    return torch.FloatTensor(spec).unsqueeze(0).unsqueeze(0)


def run_combined_pipeline(det_model, mic_label, cls_model,
                          test_wavs, val_dataset, device):
    """Optimise threshold on val set, then evaluate Detector->Classifier on test set."""
    det_model.eval()
    val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False,
                            num_workers=4, persistent_workers=True)
    val_probs, val_true = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            val_probs.extend(torch.softmax(det_model(inputs.to(device)), dim=1)[:, 1].cpu().numpy())
            val_true.extend(labels.numpy())
    val_probs = np.array(val_probs)
    val_true  = np.array(val_true)

    best_t, best_f1 = 0.1, 0.0  # 0.1 = first value in search grid; never an unevaluated sentinel
    for t in np.arange(0.1, 0.95, 0.05):
        preds = (val_probs >= t).astype(int)
        _, _, f1, _ = precision_recall_fscore_support(
            val_true, preds, average='binary', pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    print(f'  [{mic_label}] threshold={best_t:.2f}  val-F1={best_f1:.4f}')

    det_model.eval(); cls_model.eval()
    cpreds, clabels = [], []
    for wav_path in tqdm(test_wavs, desc=f'Pipeline {mic_label}', leave=False):
        clabels.append(CLS_CLASS_TO_IDX[get_class_from_filename(wav_path)])
        det_inp = _spec_tensor(wav_path, DET_CONFIG, det_seq_len).to(device)
        with torch.no_grad():
            prob = torch.softmax(det_model(det_inp), dim=1)[0, 1].item()
        if prob < best_t:
            cpreds.append(CLS_CLASS_TO_IDX.get('noise', 0))
        else:
            cls_inp = _spec_tensor(wav_path, CLS_CONFIG, cls_seq_len).to(device)
            with torch.no_grad():
                cpreds.append(cls_model(cls_inp).argmax(dim=1).item())

    cpreds  = np.array(cpreds)
    clabels = np.array(clabels)
    acc  = accuracy_score(clabels, cpreds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        clabels, cpreds, average='macro', zero_division=0)
    cm = confusion_matrix(clabels, cpreds)
    report_str = classification_report(clabels, cpreds, target_names=CLS_CLASSES, zero_division=0)
    return {'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1,
            'cm': cm, 'report': report_str, 'threshold': best_t}


pipeline_results = {}
for _mic, _dr in det_results.items():
    print(f'\n{"="*60}')
    print(f'COMBINED PIPELINE: Detector {_mic} -> Classifier')
    print(f'{"="*60}')
    _res = run_combined_pipeline(
        _dr['model'], _mic, cls_model, test_wavs, det_val, DEVICE)
    pipeline_results[_mic] = _res

    print(f'\n=== Combined {_mic}: Accuracy={_res["acc"]:.4f}  F1={_res["f1"]:.4f} ===')
    print(_res['report'])

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(_res['cm'], interpolation='nearest', cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    n_cls = len(CLS_CLASSES)
    ax.set(xticks=range(n_cls), yticks=range(n_cls),
           xticklabels=CLS_CLASSES, yticklabels=CLS_CLASSES,
           xlabel='Predicted', ylabel='True',
           title=f'Combined {_mic} Confusion Matrix (Test)')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    for i in range(n_cls):
        for j in range(n_cls):
            v = _res['cm'][i, j]
            ax.text(j, i, str(v), ha='center', va='center',
                    color='white' if v > _res['cm'].max()/2 else 'black')
    plt.tight_layout(); plt.show()

# Final comparison table
print(f'\n{"="*60}')
print('COMBINED PIPELINE COMPARISON SUMMARY')
print(f'{"="*60}')
print(f'{"Detector":<12} {"Threshold":>10} {"Accuracy":>10} {"Prec":>8} {"Rec":>8} {"F1":>8}')
print('-'*60)
for _mic, _r in pipeline_results.items():
    print(f'{_mic:<12} {_r["threshold"]:>10.2f} {_r["acc"]:>10.4f} '
          f'{_r["prec"]:>8.4f} {_r["rec"]:>8.4f} {_r["f1"]:>8.4f}')


In [ ]:
# Cell 15: Export Models as .pk
#
# Exports all fine-tuned detector variants + the single classifier.

def export_pk(model, encoderOpts, classifierOpts, dataOpts, classes, path):
    """Export model as .pk file compatible with BatSpot GUI/CLI."""
    if isinstance(model, nn.DataParallel):
        model = model.module
    model = model.cpu()
    encoder = model[0]
    classifier = model[1]
    torch.save({
        'encoderOpts':     encoderOpts,
        'classifierOpts':  classifierOpts,
        'dataOpts':        dataOpts,
        'encoderState':    encoder.state_dict(),
        'classifierState': classifier.state_dict(),
        'classes':         classes,
    }, path)
    print(f'Saved: {path} ({os.path.getsize(path)/1e6:.1f} MB)')

det_dataOpts = {
    'sr': DET_CONFIG['sr'], 'n_fft': DET_CONFIG['n_fft'],
    'hop_length': DET_CONFIG['hop_length'], 'n_freq_bins': DET_CONFIG['n_freq_bins'],
    'fmin': DET_CONFIG['fmin'], 'fmax': DET_CONFIG['fmax'],
    'freq_compression': DET_CONFIG['freq_compression'],
    'min_level_db': -100, 'ref_level_db': 20, 'preemphases': 0.98,
}
det_classes = {'noise': 0, 'call': 1}

# Export each detector variant
det_out_paths = {}
for _mic, _dr in det_results.items():
    _out = os.path.join(WORKING_DIR, f'detector_192khz_{_mic}.pk')
    export_pk(_dr['model'], _dr['encoderOpts'], _dr['classifierOpts'],
              det_dataOpts, det_classes, _out)
    det_out_paths[_mic] = _out

# Export classifier (250kHz, multi-species)
cls_dataOpts = {
    'sr': CLS_CONFIG['sr'], 'n_fft': CLS_CONFIG['n_fft'],
    'hop_length': CLS_CONFIG['hop_length'], 'n_freq_bins': CLS_CONFIG['n_freq_bins'],
    'fmin': CLS_CONFIG['fmin'], 'fmax': CLS_CONFIG['fmax'],
    'freq_compression': CLS_CONFIG['freq_compression'],
    'min_level_db': -100, 'ref_level_db': 20, 'preemphases': 0.98,
}
cls_out_path = os.path.join(WORKING_DIR, 'classifier_250khz.pk')
export_pk(cls_model, encoderOpts_cls, cls_classifierOpts, cls_dataOpts, CLS_CLASS_TO_IDX, cls_out_path)

# Verify exports
print('\nVerifying exports...')
for _mic, p in det_out_paths.items():
    obj = torch.load(p, map_location='cpu', weights_only=False)
    print(f'  detector_{_mic}: enc={len(obj["encoderState"])} params, '
          f'cls={len(obj["classifierState"])} params, classes={obj["classes"]}')
obj = torch.load(cls_out_path, map_location='cpu', weights_only=False)
print(f'  classifier:    enc={len(obj["encoderState"])} params, '
      f'cls={len(obj["classifierState"])} params, classes={list(obj["classes"].keys())}')


In [ ]:
# Cell 16: Model Inventory — official pre-trained models + your own models

if MODEL_OUTPUT_DIR is None:
    for candidate in ['Data/model_output', '/kaggle/input/model_output',
                      '/kaggle/input/batspot-data/model_output']:
        if os.path.isdir(candidate):
            MODEL_OUTPUT_DIR = candidate
            break

model_inventory = []
model_tags = {}

if MODEL_OUTPUT_DIR and os.path.isdir(MODEL_OUTPUT_DIR):
    for root, dirs, files in os.walk(MODEL_OUTPUT_DIR):
        for f in files:
            if f.endswith('.pk') or f.endswith('.checkpoint'):
                p = os.path.join(root, f)
                model_inventory.append(p)
                model_tags[p] = 'yours'

# Official models staged by Cell 10
for role, mics in PRETRAINED_MODELS.items():
    for mic, info in sorted(mics.items()):
        p = info['path']
        if p not in model_inventory:
            model_inventory.append(p)
        model_tags[p] = f'official:{role}/{mic}'

print(f'MODEL_OUTPUT_DIR: {MODEL_OUTPUT_DIR}')
print(f'Found {len(model_inventory)} models:\n')
for m in model_inventory:
    tag = model_tags.get(m, '?')
    size_mb = os.path.getsize(m) / 1e6
    fmt = '.pk' if m.endswith('.pk') else '.checkpoint'
    info = None
    for role, mics in PRETRAINED_MODELS.items():
        for mic, inf in mics.items():
            if inf['path'] == m:
                info = inf
    if info:
        print(f'  [{tag:22s}] {size_mb:6.1f} MB {fmt}  '
              f'{info["num_classes"]} classes, {(info["sr"] or 0)//1000} kHz')
    else:
        print(f'  [{tag:22s}] {size_mb:6.1f} MB {fmt}  {os.path.basename(m)}')


In [ ]:
# Cell 17: Load & Evaluate Each Existing Model

def run_model_evaluation(model_path, wav_files, class_to_idx, device):
    """Evaluate a single model on the dataset."""
    model_name = os.path.basename(model_path)
    try:
        model, model_classes, dataOpts = load_any_model(model_path, device)
    except Exception as e:
        return {'name': model_name, 'path': model_path, 'error': str(e)}

    model.eval()
    model.to(device)

    model_class_names = {v: k for k, v in model_classes.items()}
    known_classes = set(model_classes.keys())
    target_to_idx = dict(model_classes)
    target_names = [model_class_names[i] for i in range(len(model_class_names))]
    eval_labels = list(range(len(target_names)))

    known_files = [f for f in wav_files if get_class_from_filename(f) in known_classes]
    data_classes = sorted({get_class_from_filename(f) for f in wav_files})
    if not known_files:
        return {
            'name': model_name, 'path': model_path,
            'error': (f'class mismatch: model knows {sorted(known_classes)}, '
                      f'dataset has {data_classes} -> 0 overlapping files'),
            'unsupported': True,
        }

    model_sr   = dataOpts.get('sr', 384000)
    model_nfft = dataOpts.get('n_fft', 256)
    model_hop  = dataOpts.get('hop_length', 128)
    model_nfreq= dataOpts.get('n_freq_bins', 256)
    model_fmin = dataOpts.get('fmin', 18000)
    model_fmax = dataOpts.get('fmax', 90000)
    seq_len = int(float(20) / 1000 * model_sr / model_hop)

    dataset = BatDataset(known_files, target_to_idx, model_sr,
                          model_nfft, model_hop, model_nfreq, model_fmin, model_fmax,
                          seq_len, dataOpts.get('freq_compression', 'linear'))
    loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            output = model(inputs)
            preds = output.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, labels=eval_labels,
                                   target_names=target_names, zero_division=0,
                                   output_dict=True)
    cm = confusion_matrix(all_labels, all_preds, labels=eval_labels)

    return {
        'name': model_name, 'path': model_path,
        'accuracy': acc, 'report': report, 'confusion_matrix': cm,
        'target_names': target_names, 'labels': all_labels, 'preds': all_preds,
        'model_classes': model_classes, 'n_files': len(known_files),
        'skipped_classes': sorted(set(data_classes) - known_classes),
    }

# Evaluate all models
val_results = []
for m_path in model_inventory:
    result = run_model_evaluation(m_path, all_wavs, CLS_CLASS_TO_IDX, DEVICE)
    val_results.append(result)
    tag = model_tags.get(m_path, '?')
    if 'error' in result:
        status = 'UNSUPPORTED' if result.get('unsupported') else 'ERROR'
        print(f'[{status}] {tag} / {result["name"]}: {result["error"]}')
    else:
        skipped = result.get('skipped_classes') or []
        extra = f', skipped_classes={skipped}' if skipped else ''
        print(f'[OK] {tag} / {result["name"]}: acc={result["accuracy"]:.4f}, '
              f'files={result["n_files"]}, classes={result["target_names"]}{extra}')


In [ ]:
# Cell 18: Results Summary & Confusion Matrices

print('\n' + '='*80)
print('EXISTING MODEL VALIDATION SUMMARY')
print('='*80)

for r in val_results:
    if 'error' in r:
        print(f'\n{r["name"]}: ERROR - {r["error"]}')
        continue
    print(f'\n--- {r["name"]} ---')
    print(f'  Files evaluated: {r["n_files"]}')
    print(f'  Classes: {r["target_names"]}')
    print(f'  Accuracy: {r["accuracy"]:.4f}')
    print(f'  Per-class:')
    for cls_name in r['target_names']:
        p = r['report'].get(cls_name, {})
        print(f'    {cls_name}: P={p.get("precision",0):.3f} R={p.get("recall",0):.3f} F1={p.get("f1-score",0):.3f} n={int(p.get("support",0))}')

    cm = r['confusion_matrix']
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    names = r['target_names']
    n = len(names)
    ax.set(xticks=range(n), yticks=range(n), xticklabels=names, yticklabels=names,
           xlabel='Predicted', ylabel='True', title=f'{r["name"][:40]}')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    for i in range(n):
        for j in range(n):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
    plt.tight_layout()
    plt.show()


In [ ]:
# Cell 19: Example Predictions from Each Model

for r in val_results:
    if 'error' in r:
        continue

    model_class_names = {v: k for k, v in r['model_classes'].items()}
    known_files = [f for f in all_wavs if get_class_from_filename(f) in r['model_classes']]

    print(f'\n--- {r["name"]} ---')
    print(f'Known classes: {r["target_names"]}')

    n_show = min(8, len(r['preds']))
    indices = np.random.choice(len(r['preds']), n_show, replace=False)
    for idx in sorted(indices):
        true_cls = model_class_names.get(r['labels'][idx], '?')
        pred_cls = model_class_names.get(r['preds'][idx], '?')
        fname = os.path.basename(known_files[idx])
        match = 'OK' if r['labels'][idx] == r['preds'][idx] else 'MISS'
        print(f'  {match} true={true_cls:6s} pred={pred_cls:6s}  {fname}')
